In [1]:
# ============================================================================
# COMPLETE TRAINING AND PREDICTION PIPELINE FOR TELECOM CHURN
# ============================================================================

import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, f1_score
import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import VotingClassifier
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def clean_feature_names(columns):
    """Clean feature names to remove special characters for LightGBM compatibility"""
    cleaned = []
    for col in columns:
        cleaned_name = re.sub(r'[^A-Za-z0-9_]', '_', str(col))
        cleaned_name = re.sub(r'_+', '_', cleaned_name)
        cleaned_name = cleaned_name.strip('_')
        cleaned.append(cleaned_name)
    return cleaned

def create_features(df, is_train=True):
    """
    Create features from the telecom dataset
    """
    data = df.copy()
    
    # 1. ARPU and billing features
    data['arpu_per_year'] = data['arpu'] / 12
    data['avg_bill_per_mou'] = data['l3m_avg_bill_dura'] / (data['l3m_avg_mou'] + 1)
    data['bill_usage_ratio'] = data['cm_tot_bill_dura'] / (data['l3m_avg_bill_dura'] + 1)
    
    # 2. Data usage features
    data['flux_usage_ratio'] = data['cm_flux_use'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    data['4g_usage_ratio'] = data['flux_4g_use'] / (data['cm_flux_use'] + 1)
    data['upload_download_ratio'] = data['flux_up_4g_sum'] / (data['flux_down_4g_sum'] + 1)
    data['avg_daily_flux'] = data['cm_flux_use'] / (data['gprs_days'] + 1)
    
    # 3. Time-based usage patterns
    data['wday_flux_total'] = data['wday_day_flux'] + data['wday_night_flux']
    data['nwday_flux_total'] = data['nwday_day_flux'] + data['nwday_night_flux']
    data['wday_vs_nwday'] = data['wday_flux_total'] / (data['nwday_flux_total'] + 1)
    data['day_vs_night_wday'] = data['wday_day_flux'] / (data['wday_night_flux'] + 1)
    data['day_vs_night_nwday'] = data['nwday_day_flux'] / (data['nwday_night_flux'] + 1)
    
    # 4. Over-plan usage
    data['has_over_plan'] = (data['cm_over_plan_flux'] > 0).astype(int)
    data['over_plan_ratio'] = data['cm_over_plan_flux'] / (data['cm_base_plan_flux'] + data['cm_chos_plan_flux'] + 1)
    
    # 5. Voice usage features
    data['local_voice_ratio'] = data['cm_local_voice_dura'] / (data['cm_tot_bill_dura'] + 1)
    data['voice_data_ratio'] = data['cm_tot_bill_dura'] / (data['cm_flux_use'] + 1)
    
    # 6. Broadband features
    if 'bd_flux_m' in data.columns:
        data['has_broadband'] = (data['bd_flux_m'] > 0).astype(int)
        data['bd_usage_intensity'] = data['bd_flux_m'] / (data['bd_dur_m'] + 1)
        data['bd_avg_session_dur'] = data['bd_dur_m'] / (data['bd_cnt_m'] + 1)
        data['mobile_bd_ratio'] = data['cm_flux_use'] / (data['bd_flux_m'] + 1)
    
    # 7. TV and entertainment features
    if 'user_duration_m' in data.columns:
        data['tv_engagement'] = data['user_duration_m'] / (data['login_times_m'] + 1)
        data['click_watch_ratio'] = data['click_times_m'] / (data['watch_times_m'] + 1)
        data['active_days_ratio'] = data['open_day_m'] / 30
    
    # 8. Video usage features
    video_cols = ['gm_use_dur', 'shrt_vid_use_dur', 'long_vid_use_dur', 
                  'anchor_use_dur', 'wtch_liv_use_dur', 'netdisk_use_dur']
    if all(col in data.columns for col in video_cols):
        data['total_video_dur'] = data[video_cols].sum(axis=1)
        data['game_video_ratio'] = data['gm_use_dur'] / (data['total_video_dur'] + 1)
        data['short_long_ratio'] = data['shrt_vid_use_dur'] / (data['long_vid_use_dur'] + 1)
        
        # Day vs night patterns
        data['game_day_ratio'] = data['gm_dayt_use_dur'] / (data['gm_use_dur'] + 1)
        data['video_night_usage'] = (data['shrt_vid_ngt_use_dur'] + 
                                      data['long_vid_ngt_use_dur'] + 
                                      data['anchor_ngt_use_dur'])
        data['video_day_usage'] = (data['shrt_vid_dayt_use_dur'] + 
                                    data['long_vid_dayt_use_dur'] + 
                                    data['anchor_dayt_use_dur'])
        data['video_day_night_ratio'] = data['video_day_usage'] / (data['video_night_usage'] + 1)
    
    # 9. Content consumption features
    content_cols = ['video_cnt_m', 'read_cnt_m', 'music_cnt_m', 'caijing_cnt_m', 
                    'travel_cnt_m', 'game_cnt_m', 'edu_cnt_m']
    if all(col in data.columns for col in content_cols):
        data['total_content_cnt'] = data[content_cols].sum(axis=1)
        data['content_diversity'] = (data[content_cols] > 0).sum(axis=1)
        data['avg_read_time'] = data['read_time_m'] / (data['read_cnt_m'] + 1)
        data['edu_engagement'] = data['edu_time_m'] / (data['edu_cnt_m'] + 1)
    
    # 10. User labels aggregation
    label_cols = ['hi_flux_usr_lbl', 'sev_vid_usr_lbl', 'liv_usr_lbl', 
                  'netdisk_usr_lbl', 'vid_usr_lbl', 'read_usr_lbl', 
                  'gm_usr_lbl', 'msc_usr_lbl']
    if all(col in data.columns for col in label_cols):
        data['total_user_labels'] = data[label_cols].sum(axis=1)
        data['is_heavy_user'] = (data['total_user_labels'] > 3).astype(int)
        data['is_video_focused'] = ((data['vid_usr_lbl'] + data['sev_vid_usr_lbl']) > 1).astype(int)
    
    # 11. Network and service features
    data['is_premium_user'] = ((data['is_fam_vnet_user'] == 1) | 
                               (data['is_ent_vnet_user'] == 1)).astype(int)
    data['has_unlimited'] = data['if_nulim_prod']
    data['service_stability'] = 1 - data['is_bd_status_abnormal']
    
    # 12. Loyalty and tenure features
    data['tenure_years'] = data['innet_dura'] / 365
    data['is_long_term'] = (data['innet_dura'] > 1095).astype(int)  # > 3 years
    data['contract_value'] = data['term_cont_mon'] * data['term_cont_dfee']
    
    # 13. Age-based features
    data['age_group'] = pd.cut(data['age'], bins=[0, 25, 35, 45, 55, 100], 
                                labels=['young', 'young_adult', 'middle', 'mature', 'senior'])
    data['is_young'] = (data['age'] < 30).astype(int)
    data['is_senior'] = (data['age'] > 55).astype(int)
    
    # 14. Out-of-network usage
    data['total_out_usage'] = data['out_gprs'] + data['out_call']
    data['out_usage_ratio'] = data['total_out_usage'] / (data['cm_flux_use'] + data['cm_tot_bill_dura'] + 1)
    
    # 15. Engagement scores
    data['engagement_score'] = (
        data['login_times_m'] * 0.2 + 
        data['click_times_m'] * 0.2 + 
        data['watch_times_m'] * 0.3 + 
        data['open_day_m'] * 0.3
    )
    
    # 16. Revenue metrics
    data['revenue_per_gb'] = data['arpu'] / ((data['cm_flux_use'] / 1024) + 1)
    data['revenue_per_minute'] = data['arpu'] / (data['l3m_avg_mou'] + 1)
    
    # Handle categorical features
    if 'age_group' in data.columns:
        data['age_group_encoded'] = LabelEncoder().fit_transform(data['age_group'].astype(str))
    
    return data

def get_feature_columns(df):
    """Get the list of feature columns to use for modeling"""
    exclude_cols = ['Unnamed: 0', 'id', 'label', 'age_group']
    feature_cols = [col for col in df.columns if col not in exclude_cols]
    return feature_cols

# ============================================================================
# LOAD AND PREPARE TRAINING DATA
# ============================================================================

print("=" * 80)
print("LOADING TRAINING DATA")
print("=" * 80)

df_train = pd.read_csv('./train.csv')
print(f"Training data shape: {df_train.shape}")
print(f"\nClass distribution:")
print(df_train['label'].value_counts())
print(f"Class ratio: {df_train['label'].value_counts()[0] / df_train['label'].value_counts()[1]:.2f}:1")

# Check for missing values
print(f"\nMissing values:")
missing_summary = df_train.isnull().sum()
missing_summary = missing_summary[missing_summary > 0].sort_values(ascending=False)
if len(missing_summary) > 0:
    print(missing_summary.head(10))
else:
    print("No missing values found")

# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

print("\n" + "=" * 80)
print("CREATING FEATURES")
print("=" * 80)

df_train_processed = create_features(df_train, is_train=True)

# Get feature columns
feature_cols = get_feature_columns(df_train_processed)
print(f"Total features after engineering: {len(feature_cols)}")

# Clean feature names
original_feature_cols = feature_cols.copy()
cleaned_feature_cols = clean_feature_names(feature_cols)
feature_name_mapping = dict(zip(cleaned_feature_cols, original_feature_cols))

# Prepare X and y
X = df_train_processed[feature_cols].copy()
X.columns = cleaned_feature_cols
y = df_train_processed['label']

# Fill missing values
print("\nHandling missing values...")
# Use median for numeric, mode for categorical
for col in X.columns:
    if X[col].dtype in ['float64', 'int64']:
        X[col].fillna(X[col].median(), inplace=True)
    else:
        X[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)

# Replace infinities
X.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Final feature matrix shape: {X.shape}")

# ============================================================================
# TRAIN-VALIDATION SPLIT
# ============================================================================

print("\n" + "=" * 80)
print("SPLITTING DATA FOR VALIDATION")
print("=" * 80)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Validation set: {X_val.shape}")
print(f"Training class distribution: {y_train.value_counts().to_dict()}")

# ============================================================================
# TRAIN MULTIPLE MODELS AND SELECT BEST
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING AND COMPARING MODELS")
print("=" * 80)

# Calculate scale_pos_weight
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale pos weight: {scale_pos_weight:.2f}")

# Model 1: LightGBM
print("\n1. Training LightGBM...")
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    force_col_wise=True,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=0.1
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_val)
y_proba_lgb = lgb_model.predict_proba(X_val)[:, 1]

lgb_acc = accuracy_score(y_val, y_pred_lgb)
lgb_auc = roc_auc_score(y_val, y_proba_lgb)
lgb_f1 = f1_score(y_val, y_pred_lgb)

print(f"LightGBM - Accuracy: {lgb_acc:.4f}, ROC-AUC: {lgb_auc:.4f}, F1: {lgb_f1:.4f}")

# Model 2: XGBoost
print("\n2. Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss',
    tree_method='hist'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_val)
y_proba_xgb = xgb_model.predict_proba(X_val)[:, 1]

xgb_acc = accuracy_score(y_val, y_pred_xgb)
xgb_auc = roc_auc_score(y_val, y_proba_xgb)
xgb_f1 = f1_score(y_val, y_pred_xgb)

print(f"XGBoost - Accuracy: {xgb_acc:.4f}, ROC-AUC: {xgb_auc:.4f}, F1: {xgb_f1:.4f}")

# Model 3: Ensemble
print("\n3. Training Ensemble...")
ensemble = VotingClassifier(
    estimators=[
        ('lgb', lgb_model),
        ('xgb', xgb_model)
    ],
    voting='soft',
    n_jobs=-1
)
ensemble.fit(X_train, y_train)
y_pred_ens = ensemble.predict(X_val)
y_proba_ens = ensemble.predict_proba(X_val)[:, 1]

ens_acc = accuracy_score(y_val, y_pred_ens)
ens_auc = roc_auc_score(y_val, y_proba_ens)
ens_f1 = f1_score(y_val, y_pred_ens)

print(f"Ensemble - Accuracy: {ens_acc:.4f}, ROC-AUC: {ens_auc:.4f}, F1: {ens_f1:.4f}")

# Select best model
models = {
    'LightGBM': (lgb_model, lgb_auc, y_pred_lgb, y_proba_lgb),
    'XGBoost': (xgb_model, xgb_auc, y_pred_xgb, y_proba_xgb),
    'Ensemble': (ensemble, ens_auc, y_pred_ens, y_proba_ens)
}

best_model_name = max(models, key=lambda k: models[k][1])
best_model, best_auc, best_pred, best_proba = models[best_model_name]

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name} (ROC-AUC: {best_auc:.4f})")
print(f"{'='*80}")
print("\nDetailed Classification Report:")
print(classification_report(y_val, best_pred))

# ============================================================================
# TRAIN FINAL MODEL ON FULL DATA
# ============================================================================

print("\n" + "=" * 80)
print("TRAINING FINAL MODEL ON FULL TRAINING DATA")
print("=" * 80)

if best_model_name == 'LightGBM':
    final_model = lgb.LGBMClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        num_leaves=31,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1,
        verbose=-1,
        force_col_wise=True,
        min_child_samples=20,
        reg_alpha=0.1,
        reg_lambda=0.1
    )
elif best_model_name == 'XGBoost':
    final_model = xgb.XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        n_jobs=-1,
        eval_metric='logloss',
        tree_method='hist'
    )
else:
    final_model = ensemble

final_model.fit(X, y)
print("Final model trained successfully!")

# ============================================================================
# FEATURE IMPORTANCE
# ============================================================================

print("\n" + "=" * 80)
print("TOP 30 MOST IMPORTANT FEATURES")
print("=" * 80)

if best_model_name in ['LightGBM', 'XGBoost']:
    feature_importance = pd.DataFrame({
        'feature': [feature_name_mapping.get(col, col) for col in X.columns],
        'importance': final_model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(feature_importance.head(30))
else:
    print("Feature importance not available for ensemble model")

# ============================================================================
# LOAD AND PREDICT TEST DATA
# ============================================================================

print("\n" + "=" * 80)
print("LOADING TEST DATA")
print("=" * 80)

df_test = pd.read_csv('./testA.csv')
print(f"Test data shape: {df_test.shape}")

# Store IDs
test_ids = df_test['id'].copy()

# Create features
print("\nCreating features for test data...")
df_test_processed = create_features(df_test, is_train=False)

# Prepare test features
X_test = df_test_processed[feature_cols].copy()
X_test.columns = cleaned_feature_cols

# Fill missing values
print("Handling missing values in test data...")
for col in X_test.columns:
    if col in X.columns:
        if X_test[col].dtype in ['float64', 'int64']:
            X_test[col].fillna(X[col].median(), inplace=True)
        else:
            X_test[col].fillna(X[col].mode()[0] if len(X[col].mode()) > 0 else 0, inplace=True)
    else:
        X_test[col].fillna(0, inplace=True)

# Replace infinities
X_test.replace([np.inf, -np.inf], 0, inplace=True)

print(f"Test feature matrix shape: {X_test.shape}")

# ============================================================================
# MAKE PREDICTIONS
# ============================================================================

print("\n" + "=" * 80)
print("MAKING PREDICTIONS")
print("=" * 80)

predictions = final_model.predict(X_test)
predictions_proba = final_model.predict_proba(X_test)[:, 1]

print(f"Predictions distribution:")
print(f"Class 0: {(predictions == 0).sum()} ({(predictions == 0).sum() / len(predictions) * 100:.1f}%)")
print(f"Class 1: {(predictions == 1).sum()} ({(predictions == 1).sum() / len(predictions) * 100:.1f}%)")

# ============================================================================
# SAVE RESULTS
# ============================================================================

print("\n" + "=" * 80)
print("SAVING RESULTS")
print("=" * 80)

# Create submission dataframe
submission = pd.DataFrame({
    'id': test_ids,
    'label': predictions
})

# Save to CSV
output_path = './submit.csv'
submission.to_csv(output_path, index=False)

print(f"Submission saved to: {output_path}")
print(f"Submission shape: {submission.shape}")
print(f"\nFirst 10 predictions:")
print(submission.head(10))

# Also save with probabilities
submission_with_proba = pd.DataFrame({
    'id': test_ids,
    'label': predictions,
    'probability': predictions_proba
})

output_path_proba = './submit_with_probabilities.csv'
submission_with_proba.to_csv(output_path_proba, index=False)
print(f"\nPredictions with probabilities saved to: {output_path_proba}")

print("\n" + "=" * 80)
print("PIPELINE COMPLETE!")
print("=" * 80)
print(f"Best Model: {best_model_name}")
print(f"Validation ROC-AUC: {best_auc:.4f}")
print(f"Total Features Used: {len(feature_cols)}")

LOADING TRAINING DATA
Training data shape: (59904, 88)

Class distribution:
label
0    44904
1    15000
Name: count, dtype: int64
Class ratio: 2.99:1

Missing values:
gm_use_dur               47451
gm_dayt_use_dur          47451
gm_ngt_use_dur           47451
shrt_vid_use_dur         47451
shrt_vid_dayt_use_dur    47451
shrt_vid_ngt_use_dur     47451
long_vid_use_dur         47451
long_vid_dayt_use_dur    47451
long_vid_ngt_use_dur     47451
anchor_use_dur           47451
dtype: int64

CREATING FEATURES
Total features after engineering: 136

Handling missing values...
Final feature matrix shape: (59904, 136)

SPLITTING DATA FOR VALIDATION
Training set: (47923, 136)
Validation set: (11981, 136)
Training class distribution: {0: 35923, 1: 12000}

TRAINING AND COMPARING MODELS
Scale pos weight: 2.99

1. Training LightGBM...
LightGBM - Accuracy: 0.7777, ROC-AUC: 0.8457, F1: 0.6258

2. Training XGBoost...
XGBoost - Accuracy: 0.7827, ROC-AUC: 0.8462, F1: 0.6290

3. Training Ensemble...
Ensemb